# Week 2 Task 2.5 — Advanced Models

Objective: compare leakage-safe XGBoost, LightGBM, and CatBoost candidates against the frozen Task 2.4 baselines, then identify—but not package—the strongest advanced candidate.

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
metrics = json.loads((ROOT/'reports/week2_advanced_metrics.json').read_text())
metrics['status'], metrics['environment']

## Baseline recap and exact chronological split

The experiment imports the exact Task 2.4 target, 42 predictors, exclusions, categorical definitions, seed 42, and time boundaries. Training ends 2016-08-31, validation spans September–October, and test spans November–December. There is no shuffle or overlap; mappings and fitting use training data only.

In [ ]:
pd.DataFrame(metrics['splits']).T

## Target skew, transformation, and drift

Raw-target and `log1p` training are screened separately. Log predictions are returned with `expm1`; negative inverse predictions alone are clipped to zero, and their pre-clip percentage is retained. All primary metrics remain on the original meter-reading scale. Split statistics demonstrate a tiny but influential extreme tail and major late-period variance drift.

In [ ]:
pd.DataFrame(metrics['drift_analysis']['target_statistics']).T

## Categorical handling and bounded experiments

`meter` and `primary_use` follow the baseline predictor contract. XGBoost receives training-fitted ordinal codes, LightGBM receives training-defined categorical levels, and CatBoost receives native string categoricals. No target encoding is used. Six screening experiments use the same two-million evenly spaced training rows and 500,000 evenly spaced validation rows; order and full time span are preserved. The selected strategy from each family is refitted on all 13,358,457 training rows with early stopping.

In [ ]:
experiments = pd.json_normalize(metrics['experiments'])
experiments[['experiment_id','model','target_strategy','training_rows','best_iteration',
             'validation.rmse','validation.mae','validation.r2','validation.rmsle',
             'training_duration_seconds']]

## XGBoost, LightGBM, and CatBoost results

The cells below expose the six screening records and the frozen full-data family finalists. Test metrics were computed only after each family target strategy had been selected from validation.

In [ ]:
rows=[]
for name, info in metrics['comparison'].items():
    rows.append({'model':name,'target':info['target_strategy'],'coverage':info['training_coverage'],
                 **{f'val_{k}':v for k,v in info['validation'].items()},
                 **{f'test_{k}':v for k,v in info['test'].items()},
                 'train_seconds':info['training_time_seconds'],
                 'test_inference_seconds':info['test_inference_seconds'],
                 'artifact_bytes':info['artifact_size_bytes']})
comparison=pd.DataFrame(rows).set_index('model')
comparison[['target','coverage','val_rmse','val_mae','val_r2','val_rmsle',
            'test_rmse','test_mae','test_r2','test_rmsle','train_seconds',
            'test_inference_seconds','artifact_bytes']]

## Error, subgroup, and drift analysis

The metrics artifact contains validation/test errors by month, meter, site, primary use, target decile, zero/nonzero rows, high-consumption rows, and building-level summaries. These diagnostics show that rare extreme observations dominate RMSE and explain much of the validation/test divergence; no outlier was removed.

In [ ]:
leader=metrics['selection']['best_advanced_candidate']
subgroups=metrics['final_candidates'][leader]['subgroup_metrics']
pd.DataFrame(subgroups['test']['by_target_decile']).T

## Feature importance

XGBoost and LightGBM plots use native gain; CatBoost uses prediction-values-change. These measures are associative and can favor flexible predictors. SHAP was omitted because it was unavailable and impractical relative to the bounded task scope.

In [ ]:
for name, info in metrics['final_candidates'].items():
    print(name, info['importance_type'])
    display(Image(filename=str(ROOT/info['feature_importance_plot'])))

## Model-selection rationale and Task 2.6 recommendation

LightGBM-log is the best advanced validation candidate. CatBoost-log has the smallest advanced test RMSE, but test performance did not drive selection. The advanced improvement over Random Forest is not meaningful overall: LightGBM improves validation RMSE only slightly while worsening validation MAE and test RMSE. Therefore Random Forest remains the safer Task 2.6 packaging recommendation, while LightGBM is retained as the strongest advanced challenger. Task 2.6 should package the chosen frozen artifact without retraining and preserve its preprocessing/category contract.

## Reproduction

Reusable implementation: `scripts/week2_advanced_models.py`. Run `python scripts/week2_advanced_models.py --n-jobs 6` for the documented experiment. Quick pipeline verification is available with `--quick-check --no-full-train`; it must not be reported as a final full-data result.